# 🗺️ Accident EDA & Blackspot Hotspot Generator

**Part of:** SafeVisionAI · IIT Madras Road Safety Hackathon 2026
**Output:** `accidents_summary.json` + `blackspot_seed.csv` → seeded to the backend database

This notebook processes the **Kaggle India Road Accidents dataset** (1M+ rows)
to produce two key intelligence artifacts:

1. **`accidents_summary.json`** — National total + top 10 states by accident count
2. **`blackspot_seed.csv`** — GPS clusters with accident counts for map hotspot visualization

---
### 📊 Dataset
- **Source:** Kaggle India Road Accidents dataset
- **Size:** ~1,048,575 rows · 30+ columns
- **Acquired via:** `setup_kaggle.ps1` + `scripts/data/seed_blackspots.py`

### 🔄 Pipeline
```
Raw CSV → Normalize columns → State summary → GPS cluster → blackspot_seed.csv
```

## 📂 Step 0 — Upload Accidents Dataset

Upload `kaggle_india_accidents.csv` from:
```
chatbot_service/data/accidents/kaggle_india_accidents.csv
```

> ⚠️ This file is ~450MB. The Hub stores it via Git LFS.

In [ ]:
# Cell 0 — Upload Dataset
from google.colab import files
print("▶ UPLOAD your accidents CSV dataset NOW:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"✅ Uploaded {filename}")


▶ UPLOAD your accidents CSV dataset NOW:


Saving kaggle_india_accidents.csv to kaggle_india_accidents.csv
✅ Uploaded kaggle_india_accidents.csv


## 📖 Step 1 — Load & Normalize Dataset

Reads the CSV and normalizes all column names to lowercase snake_case.
Result: **1,048,575 rows** of accident records across Indian states.

> 💡 The mixed-type DtypeWarning is expected for columns with mixed numeric/string data.

In [ ]:
# Cell 1 — Read baseline datasets
import pandas as pd, json
df = pd.read_csv(filename)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(f"Loaded accidents dataset with {len(df)} rows.")


/tmp/ipykernel_3268/1948449410.py:3: DtypeWarning: Columns (8,10,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


Loaded accidents dataset with 1048575 rows.


## 📊 Step 2 — Generate National Summary JSON

Auto-detects the `state` and `accident` columns using flexible column name matching,
then computes:
- **National total** — sum of all accident counts
- **Top 10 states** — ranked by accident volume

Exports `accidents_summary.json` — used by the chatbot to answer national stats queries.

In [ ]:
# Cell 2 — Generate Summary JSON
state_col = next((c for c in df.columns if 'state' in c), None)
acc_col = next((c for c in df.columns if 'accident' in c), df.columns[2])

summary = { "national_total": int(df[acc_col].sum()) }
if state_col:
    by_state = df.groupby(state_col)[acc_col].sum().sort_values(ascending=False).head(10)
    summary['top_states'] = [{'state': s, 'accidents': int(a)} for s, a in by_state.items()]

with open('accidents_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

from google.colab import files
files.download('accidents_summary.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 📍 Step 3 — Generate GPS Blackspot Clusters

Groups accident records by rounded GPS coordinates (2 decimal places ≈ ~1km²),
then counts accidents per grid cell.

Result: **4,134 blackspot clusters** exported as `blackspot_seed.csv`
→ This CSV is loaded by `backend/scripts/app/seed_emergency.py` to populate the PostGIS accident layer.

| Column | Description |
|--------|-------------|
| `lat_r` | Rounded latitude (±0.01°) |
| `lon_r` | Rounded longitude (±0.01°) |
| `accident_count` | Number of accidents in this 1km² cell |

In [ ]:
# Cell 3 — Process raw GPS tags into hotspot clusters
lat_col = next((c for c in df.columns if 'lat' in c), None)
lon_col = next((c for c in df.columns if 'lon' in c or 'lng' in c), None)

if lat_col and lon_col:
    df['lat_r'] = df[lat_col].round(2)
    df['lon_r'] = df[lon_col].round(2)

    hotspots = df.groupby(['lat_r','lon_r']).agg(accident_count=(acc_col, 'sum')).reset_index()
    hotspots = hotspots[hotspots['accident_count'] > 0]

    hotspots.to_csv('blackspot_seed.csv', index=False)
    print(f"✅ Generated blackspot_seed.csv with {len(hotspots)} clusters")

    files.download('blackspot_seed.csv')
else:
    print("⚠️ No Latitude/Longitude column found in dataset, skipping cluster generation.")


✅ Generated blackspot_seed.csv with 4134 clusters


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>